In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import joblib
import numpy as np
import pandas as pd

# Load model & metadata

In [5]:
folder_path = "/content/drive/MyDrive/Predicting-the-results-of-a-match/models"

In [6]:
model = joblib.load(folder_path+"/best/logistic_regression.pkl")
feature_names = joblib.load(folder_path+"/feature_names/feature_names.pkl")
label_mapping = joblib.load(folder_path+"/label_mapping/label_mapping.pkl")

In [7]:
inv_label_mapping = {v: k for k, v in label_mapping.items()}

# FastAPI app

In [8]:
app = FastAPI(
    title="Football Match Outcome Prediction API",
    description="Professional ML API for predicting football match outcomes",
    version="1.0.0"
)

# Input schema

In [9]:
class MatchFeatures(BaseModel):
    home_shots: float
    home_shotsOnTarget: float
    home_ppda: float
    away_shots: float
    away_shotsOnTarget: float
    away_ppda: float

# Health check

In [10]:
@app.get("/")
def health_check():
    return {
        "status": "API is running",
        "model": "Logistic Regression",
        "features": feature_names
    }



---
# Prediction endpoint
---



In [12]:
@app.post("/predict")
def predict_match(features: MatchFeatures):

    try:
        # Convert input to DataFrame
        input_df = pd.DataFrame([[
            features.home_shots,
            features.home_shotsOnTarget,
            features.home_ppda,
            features.away_shots,
            features.away_shotsOnTarget,
            features.away_ppda
        ]], columns=feature_names)

        # Prediction
        pred_class = model.predict(input_df)[0]
        pred_proba = model.predict_proba(input_df)[0]

        return {
            "prediction": inv_label_mapping[pred_class],
            "probabilities": {
                inv_label_mapping[i]: round(prob, 4)
                for i, prob in enumerate(pred_proba)
            }
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

<h1>Lancer l'API</h1>

```
pip install fastapi uvicorn scikit-learn numpy pydantic
uvicorn app:app --reload
```

